# Silver: workers

Preparação dos dados da bronze antes de salvar na silver. Sem cálculos de negócio: só tipagem, padronização, deduplicação e validação.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [0]:
SOURCE_TABLE = "projetos.bronze.workers"
TARGET_TABLE = "projetos.silver.tab_workers"

REQUIRED_COLUMNS = [
    "worker_id", "worker_type", "worker_status", "hire_date",
    "country", "company", "base_pay_annual", "currency", "manager_id",
]

In [0]:
df = spark.table(SOURCE_TABLE)

## Data Quality

In [0]:
rename_map = {
    "Worker_ID": "worker_id",
    "Worker_Type": "worker_type",
    "First_Name": "first_name",
    "Last_Name": "last_name",
    "Legal_Name": "legal_name",
    "Email_Work": "email_work",
    "Phone_Number": "phone_number",
    "Gender": "gender",
    "Date_of_Birth": "date_of_birth",
    "National_ID_Type": "national_id_type",
    "National_ID": "national_id",
    "Country": "country",
    "City": "city",
    "Location": "location",
    "Company": "company",
    "Cost_Center": "cost_center",
    "Supervisory_Organization": "supervisory_organization",
    "Department": "department",
    "Job_Profile": "job_profile",
    "Business_Title": "business_title",
    "Job_Family": "job_family",
    "Management_Level": "management_level",
    "Manager_ID": "manager_id",
    "Worker_Status": "worker_status",
    "Employee_Type": "employee_type",
    "Time_Type": "time_type",
    "Pay_Rate_Type": "pay_rate_type",
    "Scheduled_Weekly_Hours": "scheduled_weekly_hours",
    "Currency": "currency",
    "Base_Pay_Annual": "base_pay_annual",
    "Original_Hire_Date": "original_hire_date",
    "Hire_Date": "hire_date",
    "Termination_Date": "termination_date",
    "Termination_Reason": "termination_reason",
    "Time_Zone": "time_zone",
    "ADP_Badge_ID": "adp_badge_id",
}

existing_columns = set(df.columns)
for old_name, new_name in rename_map.items():
    if old_name in existing_columns and old_name != new_name:
        df = df.withColumnRenamed(old_name, new_name)

Padronizar textos: remove espaços e transforma "null" / vazio em NULL de verdade

In [0]:
NULL_TOKENS = ["null", "NULL", "Null", ""]

string_cols = [f.name for f in df.schema.fields
               if f.dataType.typeName() == "string" and not f.name.startswith("_")]

for c in string_cols:
    limpo = F.trim(F.col(c))
    df = df.withColumn(c, F.when(limpo.isin(NULL_TOKENS) | limpo.isNull(), F.lit(None))
                           .otherwise(limpo))

df = df.withColumn("email_work", F.lower(F.col("email_work")))

Tipagem

In [0]:
df = (
    df.withColumn("date_of_birth", F.to_date("date_of_birth", "yyyy-MM-dd"))
      .withColumn("original_hire_date", F.to_date("original_hire_date", "yyyy-MM-dd"))
      .withColumn("hire_date", F.to_date("hire_date", "yyyy-MM-dd"))
      .withColumn("termination_date", F.to_date("termination_date", "yyyy-MM-dd"))
      .withColumn("scheduled_weekly_hours", F.col("scheduled_weekly_hours").cast("decimal(5,2)"))
      .withColumn("base_pay_annual", F.col("base_pay_annual").cast("decimal(18,2)"))
)

Metadados de ingestão

In [0]:
df = (
    df.withColumn("source_file", F.col("_file_metadata")["file_name"])
      .withColumn("file_modification_time", F.col("_file_metadata")["file_modification_time"])
      .drop("_file_metadata", "_gdrive_metadata")
)

Validação: falha cedo em vez de gravar lixo na silver

In [0]:
colunas_faltando = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if colunas_faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {colunas_faltando}")

chaves_nulas = df.where(F.col("worker_id").isNull()).count()
if chaves_nulas > 0:
    raise ValueError(f"{chaves_nulas} registros sem worker_id")

## Save

In [0]:
target = DeltaTable.forName(spark, TARGET_TABLE)

(
    target.alias("target")
    .merge(df.alias("source"), "target.worker_id = source.worker_id")
    .whenMatchedUpdateAll(condition="source.file_modification_time > target.file_modification_time")
    .whenNotMatchedInsertAll()
    .execute()
)